# Explainability of Departures from the ECD Thresholds (Analysis B)

## Protocol

**Status:** fixed on 2026-10-16, before any analysis code in this notebook was run. Analysis B of `ANALYSIS_PLAN.md`, recorded in git before execution.

### 1. Framing

This is an **exploratory, descriptive analysis**.

- **No hypothesis tests** are run, and no claim of improvement is made.
- SHAP values describe **associations within a fitted model**: how the model's output for a tissue changes with its inputs. They are **not clinical explanations of specialist decisions** and **not causal evidence**. Any pattern found is a hypothesis for clinical review.

Context from notebooks 10–11:

- Improvement beyond ECD alone was not established. The hybrid model (macro F1 0.789) did not outperform a depth-2 ECD tree fitted within training folds (0.796).
- The tree recovered cut points near 2230 and 2500 in every fold.
- The two models make different errors. The hybrid recovers more grade A tissues that depart from the thresholds (A recall 0.65 vs 0.53) but loses A+ recall (0.89 vs 0.99).

### 2. Question

When the hybrid correctly predicts a tissue's recorded grade but the ECD tree does not, which feature groups are associated with the hybrid's prediction, and how does this compare with tissues where both models are correct or only the tree is correct?

### 3. Cohort and models

- **Evaluation set:** identical to notebooks 10–11. 470 non-2023 tissues with ECD, 280 donors; A+ 244, A 159, B 67. Asserted in code.
- **Folds:** the existing 5 repeats × 5 donor-grouped folds.
- **Hybrid:** the notebook 09–11 pipeline (24 tabular features + frozen ResNet-50 → PCA 16 → Random Forest), trained on all training-fold tissues of the 551-tissue primary cohort.
- **ECD tree:** the notebook 11 depth-2 tree on ECD alone, trained on training-fold tissues with ECD.
- **Reproducibility check:** the refitted Hybrid and ECD tree fold-level scores must match `fitted_ecd_fold_scores.csv` from notebook 11 (absolute difference < 1e-9). Otherwise the notebook stops and the discrepancy is logged.

### 4. Case groups

The unit is a **tissue–repeat pair**: each evaluation tissue is held out once per repeat, so it has five pairs. Using out-of-fold predictions:

| Group | Hybrid | ECD tree | Role |
|---|---|---|---|
| **Hybrid-only correct** | Correct | Wrong | Cases of interest |
| Both correct | Correct | Correct | Reference |
| Tree-only correct | Wrong | Correct | Reference: where the hybrid's departures from ECD fail |
| Both wrong | Wrong | Wrong | Counts only |

**Departure direction** compares the recorded grade with the tree's prediction, using the grade order A+ > A > B:

- **Downgrade:** recorded grade worse than the tree predicts (e.g. recorded A where the tree predicts A+).
- **Upgrade:** recorded grade better than the tree predicts.
- **None:** they agree.

**Tissue-level consistency:** for each tissue, the number of repeats (0–5) in which it falls in the hybrid-only-correct group. Report how many tissues are hybrid-only correct in at least 1, at least 3, and all 5 repeats.

### 5. SHAP procedure

- **Explainer:** `shap.TreeExplainer` on the Random Forest of the hybrid fold model that did **not** train on the tissue, with `feature_perturbation="tree_path_dependent"` and no background data.
- **Inputs explained:** the held-out tissues' features after that fold's preprocessing and PCA.
- **Output explained:** the model's predicted probability of the tissue's **recorded grade**. Positive values push the model toward the recorded grade.
- **Additivity:** for every fold, SHAP values plus the expected value must reproduce the model's predicted probabilities (maximum absolute error < 1e-6). Otherwise the notebook stops.
- **Grouping:** SHAP values are summed within groups, which preserves additivity. All transformed columns derived from a source feature, including one-hot and missing-category columns, belong to that feature's group.

| Group | Source features |
|---|---|
| ECD-derived | `ECD_CD_cells_per_mm2`, `Total_Cell_Count`, `Total_Area_um2` |
| Variability | `CV`, `SD` |
| HEX | `HEX_percent` |
| Slit-lamp: stroma | `Stroma_*` (6 features) |
| Slit-lamp: epithelium | `Epi_*` (5 features) |
| Slit-lamp: endothelium | `Endo_*` (4 features) |
| Slit-lamp: Descemet | `Descemet_*` (3 features) |
| Image | 16 PCA components of the ResNet-50 embedding |

ECD, cell count and area are grouped because ECD equals count / area. CV and SD are grouped because CV is effectively SD over mean cell area. Within-group attributions are therefore not interpreted.

### 6. Reporting (all descriptive)

1. **Group sizes:** tissue–repeat counts per case group (overall and per repeat), unique tissues, and the tissue-level consistency counts (Section 4).
2. **Composition:** case groups by recorded grade, tree prediction and departure direction.
3. **Grouped SHAP toward the recorded grade:** for each case group, mean, median and interquartile range of the signed grouped SHAP value, and mean absolute value, per feature group. Shown for all cases and split by departure direction.
4. **Stability:** the hybrid-only-correct summary (item 3) recomputed within each repeat separately; report the range across repeats and the number of repeats in which each feature group ranks first by mean signed SHAP.
5. **Finding prevalence (raw data, not SHAP):** for hybrid-only-correct downgrade cases vs both-correct cases with the same tree prediction, the proportion of tissues with each slit-lamp binary finding present, and the median of each Konan feature.
6. **Image group:** magnitudes only. Image components have no clinical labels and are not interpreted.

**Small-group rule:** any summary for a subgroup with fewer than 10 unique tissues is reported as counts only.

**Privacy rule:** outputs report aggregates only. No tissue IDs, donor attributes or individual images are displayed.

### 7. Interpretation rules

**Permitted wording:**

- "In hybrid-only-correct cases, the model's prediction toward the recorded grade was associated mainly with feature group X."
- "Feature group X contributed more / less than in both-correct cases."

**Not permitted:**

- That a feature group explains, causes or justifies the specialist's grade.
- That the hybrid "understands" the departures.
- Any claim of improvement.

**Required context:** the tree-only-correct group must be reported alongside, because the same feature groups may also push the hybrid away from correct ECD-based predictions. A group that dominates in both hybrid-only-correct and tree-only-correct cases indicates model reliance, not useful signal.

**Stability:** a pattern is described as stable only if it holds in at least 4 of 5 repeats. Otherwise it is reported as unstable.

**Other rules:**

- **No changes after results:** no other explainers, groupings, case definitions, output classes or models. Any further analysis is labelled post hoc.
- **Errors:** a code error may be fixed and the notebook rerun only if the fix does not change a protocol choice. Every rerun is logged in the deviations log with its reason.

# Analysis

Implements the protocol above without changes. Section numbers refer to the protocol. Outputs are aggregates only (Section 6, privacy rule).

In [1]:
import hashlib
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import shap
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path("../../scripts").resolve()))
import modeling_setup as ms

SEED = ms.RANDOM_STATE
ECD = "ECD_CD_cells_per_mm2"
IMAGE_DATA_DIR = ms.PROCESSED_DIR.parent.parent / "image" / "data" / "processed"
EMBEDDING_PATH = IMAGE_DATA_DIR / "embeddings" / "resnet50_imagenet_fullres.npz"
NB11_SCORES_PATH = ms.PROCESSED_DIR / "fitted_ecd_fold_scores.csv"
REVIEW_IDS = ["NEB-2024-08-16259 R", "NEB-2025-09-17471 R"]
PCA_COMPONENTS = 16
MIN_TISSUES = 10  # Section 6 small-group rule

RF_PARAMS = dict(
    n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=1,
    max_features="sqrt", class_weight="balanced", random_state=SEED, n_jobs=-1,
)
TREE_PARAMS = dict(
    max_depth=2, min_samples_leaf=10, class_weight="balanced",
    criterion="gini", random_state=SEED,
)

EXPECTED_EVAL = {
    "tissues": 470, "donors": 280,
    "grades": {"A+": 244, "A": 159, "B": 67},
}

# Section 5 feature groups.
FEATURE_GROUPS = {
    "ECD-derived": ["ECD_CD_cells_per_mm2", "Total_Cell_Count", "Total_Area_um2"],
    "Variability": ["CV", "SD"],
    "HEX": ["HEX_percent"],
    "Slit-lamp: stroma": [f for f in ms.ALL_FEATURES if f.startswith("Stroma_")],
    "Slit-lamp: epithelium": [f for f in ms.ALL_FEATURES if f.startswith("Epi_")],
    "Slit-lamp: endothelium": [f for f in ms.ALL_FEATURES if f.startswith("Endo_")],
    "Slit-lamp: Descemet": [f for f in ms.ALL_FEATURES if f.startswith("Descemet_")],
}
assert sorted(f for v in FEATURE_GROUPS.values() for f in v) == sorted(ms.ALL_FEATURES)
GROUPS = list(FEATURE_GROUPS) + ["Image"]

print("shap", shap.__version__)
for path in [ms.DATA_PATH, ms.DONOR_FOLDS_PATH, EMBEDDING_PATH, NB11_SCORES_PATH]:
    print(f"{path.name}: sha256 {hashlib.sha256(path.read_bytes()).hexdigest()[:16]}…")

shap 0.51.0
cleaned_grading_dataset.xlsx: sha256 0ce9b384a51a4c47…
donor_grouped_folds.csv: sha256 755280ff9d7bd388…
resnet50_imagenet_fullres.npz: sha256 883d4a1723a17be7…
fitted_ecd_fold_scores.csv: sha256 fe0c10e5d3431501…


## Cohort (Protocol Section 3)

In [2]:
def normalize_id(value):
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

labelled = ms.load_labelled_data()
folds = ms.get_donor_folds(labelled)

manifest = pd.read_csv(IMAGE_DATA_DIR / "clean_image_manifest.csv")
manifest["Match_ID"] = manifest["Tissue_ID"].map(normalize_id)
manifest = manifest[~manifest["Match_ID"].isin({normalize_id(i) for i in REVIEW_IDS})]

labelled["Match_ID"] = labelled["Tissue_ID"].map(normalize_id)
labelled["Year"] = labelled["Tissue_ID"].str.extract(r"NEB-\s*(\d{4})")[0].astype("Int64")

with_image = labelled.merge(manifest[["Match_ID"]], on="Match_ID", how="inner", validate="one_to_one")
cohort = (
    with_image[with_image["Year"].notna() & (with_image["Year"] != 2023)]
    .merge(folds, on="Tissue_ID", how="left", validate="one_to_one")
    .reset_index(drop=True)
)
has_ecd = cohort[ECD].notna().to_numpy()
evaluation = cohort[has_ecd]

eval_desc = {
    "tissues": len(evaluation),
    "donors": evaluation["Donor_Group"].nunique(),
    "grades": {k: int(v) for k, v in evaluation[ms.TARGET].value_counts().items()},
}
print("Primary cohort:", len(cohort), "tissues; evaluation set:", eval_desc)
assert len(cohort) == 551 and eval_desc == EXPECTED_EVAL, "Cohort differs from protocol"

cached = np.load(EMBEDDING_PATH, allow_pickle=True)
row_of = {tissue_id: i for i, tissue_id in enumerate(cached["tissue_ids"])}
embeddings = cached["embeddings"][[row_of[t] for t in cohort["Tissue_ID"]]]
EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(embeddings.shape[1])]
X = pd.concat([
    cohort[ms.ALL_FEATURES].reset_index(drop=True),
    pd.DataFrame(embeddings, columns=EMBEDDING_COLUMNS),
], axis=1)
y = cohort[ms.TARGET].to_numpy()

Primary cohort: 551 tissues; evaluation set: {'tissues': 470, 'donors': 280, 'grades': {'A+': 244, 'A': 159, 'B': 67}}


## Models (Protocol Section 3)

In [3]:
hybrid_pipeline = Pipeline([
    ("features", ColumnTransformer([
        ("tabular", ms.make_preprocessor(), ms.ALL_FEATURES),
        ("image", Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=PCA_COMPONENTS, random_state=SEED)),
        ]), EMBEDDING_COLUMNS),
    ], remainder="drop")),
    ("classifier", RandomForestClassifier(**RF_PARAMS)),
])
tree_pipeline = Pipeline([
    ("features", ColumnTransformer([("ecd", "passthrough", [ECD])], remainder="drop")),
    ("classifier", DecisionTreeClassifier(**TREE_PARAMS)),
])


def source_feature(column):
    # Map a transformed column name to its Section 5 feature group.
    if column.startswith("image__"):
        return "Image"
    remainder = column.split("__", 2)[2]
    matches = [f for f in ms.ALL_FEATURES if remainder == f or remainder.startswith(f + "_")]
    feature = max(matches, key=len)
    return next(group for group, members in FEATURE_GROUPS.items() if feature in members)

## Out-of-fold predictions and SHAP values (Protocol Sections 4–5)

In [4]:
records = []
shap_rows = []
fold_rows = []
max_additivity_error = 0.0

METRICS = {
    "Macro F1": lambda t, p: f1_score(t, p, labels=ms.CLASSES, average="macro", zero_division=0),
    "Balanced accuracy": balanced_accuracy_score,
    "Accuracy": accuracy_score,
    "Recall B": lambda t, p: recall_score(t, p, labels=["B"], average="macro", zero_division=0),
}

start = time.time()
for repeat in range(ms.N_REPEATS):
    for fold, (train, test) in enumerate(ms.iter_donor_folds(folds, cohort["Tissue_ID"], repeat)):
        assert set(cohort.loc[train, "Donor_Group"]).isdisjoint(cohort.loc[test, "Donor_Group"]), "Donor leak"
        eval_test = test[has_ecd[test]]

        hybrid = clone(hybrid_pipeline).fit(X.iloc[train], y[train])
        train_ecd = train[has_ecd[train]]
        tree = clone(tree_pipeline).fit(X.iloc[train_ecd], y[train_ecd])

        hybrid_pred = hybrid.predict(X.iloc[eval_test])
        tree_pred = tree.predict(X.iloc[eval_test])
        for name, pred in [("Hybrid", hybrid_pred), ("ECD tree", tree_pred)]:
            fold_rows.append({"Model": name, "Repeat": repeat, "Fold": fold,
                              **{m: fn(y[eval_test], pred) for m, fn in METRICS.items()}})

        # SHAP for the hybrid fold model on its held-out evaluation tissues.
        transformer = hybrid.named_steps["features"]
        forest = hybrid.named_steps["classifier"]
        X_transformed = transformer.transform(X.iloc[eval_test])
        explainer = shap.TreeExplainer(forest, feature_perturbation="tree_path_dependent")
        values = np.asarray(explainer.shap_values(X_transformed))  # (tissues, features, classes)

        reconstructed = values.sum(axis=1) + np.asarray(explainer.expected_value)
        error = np.abs(reconstructed - forest.predict_proba(X_transformed)).max()
        max_additivity_error = max(max_additivity_error, error)
        assert error < 1e-6, f"SHAP additivity failed (error {error:.2e})"

        column_groups = np.array([source_feature(c) for c in transformer.get_feature_names_out()])
        class_index = {c: i for i, c in enumerate(forest.classes_)}
        recorded_idx = np.array([class_index[g] for g in y[eval_test]])
        toward_recorded = values[np.arange(len(eval_test)), :, recorded_idx]  # (tissues, features)

        for i, row in enumerate(eval_test):
            records.append({
                "Row": row, "Repeat": repeat, "Fold": fold,
                "Recorded": y[row], "Hybrid": hybrid_pred[i], "Tree": tree_pred[i],
            })
            shap_rows.append({g: toward_recorded[i, column_groups == g].sum() for g in GROUPS})

print(f"Finished in {(time.time() - start) / 60:.1f} min; max SHAP additivity error {max_additivity_error:.2e}")
cases = pd.concat([pd.DataFrame(records), pd.DataFrame(shap_rows)], axis=1)
print("Tissue–repeat pairs:", len(cases))

Finished in 0.7 min; max SHAP additivity error 1.44e-15
Tissue–repeat pairs: 2350


### Reproducibility check (Protocol Section 3)

In [5]:
ours = pd.DataFrame(fold_rows)
saved = pd.read_csv(NB11_SCORES_PATH)
for model in ["Hybrid", "ECD tree"]:
    a = ours[ours["Model"] == model].set_index(["Repeat", "Fold"])[list(METRICS)].sort_index()
    b = saved[saved["Model"] == model].set_index(["Repeat", "Fold"])[list(METRICS)].sort_index()
    max_diff = (a - b).abs().to_numpy().max()
    print(f"{model}: max absolute difference from notebook 11 = {max_diff:.2e}")
    assert max_diff < 1e-9, f"{model} differs from notebook 11: stop and record in the deviations log"

Hybrid: max absolute difference from notebook 11 = 1.11e-16
ECD tree: max absolute difference from notebook 11 = 0.00e+00


## Case groups (Protocol Section 4)

In [6]:
RANK = {"A+": 2, "A": 1, "B": 0}

hybrid_ok = cases["Hybrid"] == cases["Recorded"]
tree_ok = cases["Tree"] == cases["Recorded"]
cases["Group"] = np.select(
    [hybrid_ok & ~tree_ok, hybrid_ok & tree_ok, ~hybrid_ok & tree_ok],
    ["Hybrid-only correct", "Both correct", "Tree-only correct"],
    default="Both wrong",
)
rank_recorded = cases["Recorded"].map(RANK)
rank_tree = cases["Tree"].map(RANK)
cases["Departure"] = np.select(
    [rank_recorded < rank_tree, rank_recorded > rank_tree], ["Downgrade", "Upgrade"], default="None"
)

CASE_GROUPS = ["Hybrid-only correct", "Both correct", "Tree-only correct", "Both wrong"]
summary = pd.DataFrame({
    "Tissue–repeat pairs": cases["Group"].value_counts(),
    "Unique tissues": cases.groupby("Group")["Row"].nunique(),
}).reindex(CASE_GROUPS)
print("1. Group sizes (all repeats):")
print(summary.to_string())
print("\nPairs per repeat:")
print(pd.crosstab(cases["Repeat"], cases["Group"])[CASE_GROUPS].to_string())

consistency = cases.assign(H=cases["Group"] == "Hybrid-only correct").groupby("Row")["H"].sum()
print("\nTissues hybrid-only correct in at least 1 repeat:", int((consistency >= 1).sum()),
      "| at least 3:", int((consistency >= 3).sum()), "| all 5:", int((consistency == 5).sum()))

1. Group sizes (all repeats):
                     Tissue–repeat pairs  Unique tissues
Group                                                   
Hybrid-only correct                  133              37
Both correct                        1739             366
Tree-only correct                    151              58
Both wrong                           327              78

Pairs per repeat:
Group   Hybrid-only correct  Both correct  Tree-only correct  Both wrong
Repeat                                                                  
0                        27           344                 34          65
1                        25           348                 30          67
2                        26           348                 30          66
3                        28           351                 27          64
4                        27           348                 30          65

Tissues hybrid-only correct in at least 1 repeat: 37 | at least 3: 30 | all 5: 14


In [7]:
print("2. Composition: case group by recorded grade and tree prediction (tissue–repeat pairs)")
print(pd.crosstab([cases["Group"], cases["Recorded"]], cases["Tree"].rename("Tree prediction"))
      .reindex(CASE_GROUPS, level=0).to_string())
print("\nCase group by departure direction:")
print(pd.crosstab(cases["Group"], cases["Departure"]).reindex(CASE_GROUPS).to_string())

2. Composition: case group by recorded grade and tree prediction (tissue–repeat pairs)
Tree prediction                 A    A+    B
Group               Recorded                
Hybrid-only correct A           0   119    0
                    A+          2     0    0
                    B          12     0    0
Both correct        A         397     0    0
                    A+          0  1084    0
                    B           0     0  258
Tree-only correct   A          23     0    0
                    A+          0   121    0
                    B           0     0    7
Both wrong          A           0   256    0
                    A+         13     0    0
                    B          28    30    0

Case group by departure direction:
Departure            Downgrade  None  Upgrade
Group                                        
Hybrid-only correct        131     0        2
Both correct                 0  1739        0
Tree-only correct            0   151        0
Both wrong       

## Grouped SHAP toward the recorded grade (Protocol Section 6, item 3)

Positive values push the hybrid toward the recorded grade. Image is reported by magnitude only and not interpreted.

In [8]:
def shap_summary(frame):
    q1, q3 = frame[GROUPS].quantile(0.25), frame[GROUPS].quantile(0.75)
    return pd.DataFrame({
        "Mean": frame[GROUPS].mean(),
        "Median": frame[GROUPS].median(),
        "IQR": [f"[{a:+.3f}, {b:+.3f}]" for a, b in zip(q1, q3)],
        "Mean |SHAP|": frame[GROUPS].abs().mean(),
    })


def report(frame, label):
    tissues = frame["Row"].nunique()
    if tissues < MIN_TISSUES:
        print(f"{label}: {len(frame)} pairs, {tissues} unique tissues — counts only (fewer than {MIN_TISSUES} tissues)\n")
        return
    print(f"{label}: {len(frame)} pairs, {tissues} unique tissues")
    print(shap_summary(frame).round(3).to_string(), "\n")


for group in CASE_GROUPS[:3]:
    report(cases[cases["Group"] == group], group)

Hybrid-only correct: 133 pairs, 37 unique tissues
                         Mean  Median               IQR  Mean |SHAP|
ECD-derived             0.029   0.035  [+0.006, +0.058]        0.043
Variability             0.027   0.023  [+0.008, +0.045]        0.032
HEX                     0.008   0.004  [-0.000, +0.017]        0.012
Slit-lamp: stroma       0.019   0.007  [+0.001, +0.012]        0.025
Slit-lamp: epithelium   0.002   0.001  [-0.005, +0.011]        0.010
Slit-lamp: endothelium -0.001  -0.001  [-0.005, +0.003]        0.006
Slit-lamp: Descemet     0.000  -0.000  [-0.005, +0.005]        0.010
Image                   0.063   0.069  [+0.016, +0.114]        0.073 

Both correct: 1739 pairs, 366 unique tissues
                         Mean  Median               IQR  Mean |SHAP|
ECD-derived             0.164   0.154  [+0.123, +0.192]        0.165
Variability             0.056   0.057  [+0.030, +0.083]        0.063
HEX                     0.004   0.003  [-0.003, +0.014]        0.011
Slit-l

In [9]:
print("By departure direction:\n")
for group in CASE_GROUPS[:3]:
    for direction in ["Downgrade", "Upgrade", "None"]:
        subset = cases[(cases["Group"] == group) & (cases["Departure"] == direction)]
        if len(subset):
            report(subset, f"{group} — {direction}")

By departure direction:

Hybrid-only correct — Downgrade: 131 pairs, 36 unique tissues
                         Mean  Median               IQR  Mean |SHAP|
ECD-derived             0.032   0.035  [+0.007, +0.058]        0.042
Variability             0.026   0.023  [+0.007, +0.044]        0.031
HEX                     0.008   0.004  [-0.000, +0.018]        0.012
Slit-lamp: stroma       0.019   0.006  [+0.001, +0.011]        0.024
Slit-lamp: epithelium   0.002   0.001  [-0.005, +0.011]        0.010
Slit-lamp: endothelium -0.001  -0.001  [-0.005, +0.003]        0.006
Slit-lamp: Descemet    -0.000  -0.000  [-0.005, +0.005]        0.010
Image                   0.062   0.067  [+0.015, +0.113]        0.072 

Hybrid-only correct — Upgrade: 2 pairs, 1 unique tissues — counts only (fewer than 10 tissues)

Both correct — None: 1739 pairs, 366 unique tissues
                         Mean  Median               IQR  Mean |SHAP|
ECD-derived             0.164   0.154  [+0.123, +0.192]        0.165
Vari

## Stability across repeats (Protocol Section 6, item 4)

A pattern is described as stable only if it holds in at least 4 of 5 repeats.

In [10]:
hybrid_only = cases[cases["Group"] == "Hybrid-only correct"]
per_repeat = hybrid_only.groupby("Repeat")[GROUPS].mean()
print("Mean signed grouped SHAP in hybrid-only-correct cases, per repeat:")
print(per_repeat.round(3).to_string())

stability = pd.DataFrame({
    "Min": per_repeat.min(), "Max": per_repeat.max(),
    "Repeats positive": (per_repeat > 0).sum(),
    "Repeats ranked first": per_repeat.idxmax(axis=1).value_counts().reindex(GROUPS, fill_value=0),
})
stability["Sign stable (≥4/5)"] = (stability["Repeats positive"] >= 4) | (stability["Repeats positive"] <= 1)
print("\nStability summary:")
print(stability.round(3).to_string())
top = stability["Repeats ranked first"].idxmax()
print(f"\nTop-ranked group: {top} in {stability.loc[top, 'Repeats ranked first']}/5 repeats "
      f"-> {'stable' if stability.loc[top, 'Repeats ranked first'] >= 4 else 'unstable'}")

Mean signed grouped SHAP in hybrid-only-correct cases, per repeat:
        ECD-derived  Variability    HEX  Slit-lamp: stroma  Slit-lamp: epithelium  Slit-lamp: endothelium  Slit-lamp: Descemet  Image
Repeat                                                                                                                               
0             0.029        0.027  0.006              0.023                  0.002                  -0.003               -0.001  0.059
1             0.024        0.029  0.005              0.029                  0.002                   0.002                0.002  0.057
2             0.039        0.020  0.007              0.014                  0.001                  -0.000               -0.001  0.072
3             0.031        0.029  0.010              0.012                  0.000                  -0.001                0.001  0.061
4             0.024        0.029  0.010              0.020                  0.003                  -0.001                0.000  0


Stability summary:
                          Min    Max  Repeats positive  Repeats ranked first  Sign stable (≥4/5)
ECD-derived             0.024  0.039                 5                     0                True
Variability             0.020  0.029                 5                     0                True
HEX                     0.005  0.010                 5                     0                True
Slit-lamp: stroma       0.012  0.029                 5                     0                True
Slit-lamp: epithelium   0.000  0.003                 5                     0                True
Slit-lamp: endothelium -0.003  0.002                 1                     0                True
Slit-lamp: Descemet    -0.001  0.002                 3                     0               False
Image                   0.057  0.072                 5                     5                True

Top-ranked group: Image in 5/5 repeats -> stable


## Finding prevalence in the raw data (Protocol Section 6, item 5)

Not SHAP. Hybrid-only-correct downgrade cases vs both-correct cases with the same tree prediction; unique tissues.

In [11]:
def prevalence(rows):
    frame = cohort.loc[sorted(rows)]
    binary = pd.Series({f: frame[f].eq(1).sum() / frame[f].notna().sum() if frame[f].notna().any() else np.nan
                        for f in ms.CLINICAL_BINARY_FEATURES})
    konan = frame[ms.NUMERIC_FEATURES].median()
    return binary, konan, len(frame)


downgrades = hybrid_only[hybrid_only["Departure"] == "Downgrade"]
for tree_grade in ["A+", "A"]:
    case_rows = set(downgrades.loc[downgrades["Tree"] == tree_grade, "Row"])
    ref_rows = set(cases.loc[(cases["Group"] == "Both correct") & (cases["Tree"] == tree_grade), "Row"])
    label = f"Tree predicts {tree_grade}"
    if len(case_rows) < MIN_TISSUES or len(ref_rows) < MIN_TISSUES:
        print(f"{label}: {len(case_rows)} downgrade tissues, {len(ref_rows)} reference tissues — counts only\n")
        continue
    case_bin, case_konan, n_case = prevalence(case_rows)
    ref_bin, ref_konan, n_ref = prevalence(ref_rows)
    print(f"{label}: hybrid-only-correct downgrades ({n_case} tissues) vs both correct ({n_ref} tissues); "
          f"{len(case_rows & ref_rows)} tissues appear in both sets across repeats")
    print(pd.DataFrame({"Downgrades": case_bin, "Both correct": ref_bin,
                        "Difference": case_bin - ref_bin}).round(3).to_string())
    print(pd.DataFrame({"Downgrades (median)": case_konan, "Both correct (median)": ref_konan}).round(1).to_string(), "\n")

Tree predicts A+: hybrid-only-correct downgrades (33 tissues) vs both correct (230 tissues); 0 tissues appear in both sets across repeats
                       Downgrades  Both correct  Difference
Stroma_Is_Clear             0.970         0.983      -0.013
Stroma_Is_Compact           0.455         0.426       0.028
Stroma_Has_Arcus            0.788         0.817      -0.030
Stroma_Has_Edema            0.061         0.013       0.048
Stroma_Has_Scar             0.152         0.000       0.152
Epi_Has_Arcus               0.697         0.546       0.151
Epi_Has_Defect              0.000         0.009      -0.009
Epi_Has_Exposure            0.576         0.507       0.069
Epi_Has_Haze                0.091         0.031       0.060
Endo_Has_Stress_Lines       0.848         0.912      -0.063
Endo_Has_RBC                0.061         0.031       0.030
Endo_Has_Cell_Dropout       0.273         0.119       0.153
Descemet_Has_Folds          0.967         0.991      -0.024
                      

In [12]:
cases.assign(Tissue_ID=cohort["Tissue_ID"].to_numpy()[cases["Row"]]).drop(columns="Row").to_csv(
    ms.PROCESSED_DIR / "threshold_departure_shap.csv", index=False
)
print("Saved per-pair grouped SHAP values to", ms.PROCESSED_DIR / "threshold_departure_shap.csv")

Saved per-pair grouped SHAP values to C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed\threshold_departure_shap.csv


## Deviations Log

Record any rerun or departure from the protocol here, with date and reason, before reporting results.

| Date | Change | Reason |
|---|---|---|
| 2026-10-16 | The protocol was **not** committed to git separately before execution, although its status line says it was recorded in git before execution. Protocol and analysis are committed together after review. | Project decision to commit only at completed milestones. The protocol text was not changed after it was written, and no analysis choice was affected. |